# GameTheory 16f - Choix Social et Solveurs SMT (z3)

**Navigation** : [<< 16d-SocialChoice-SAT](GameTheory-16d-SocialChoice-SAT.ipynb) | [Index](README.md)

**Autres side tracks** : [16b-Lean-SocialChoice](GameTheory-16b-Lean-SocialChoice.ipynb) | [16c-SocialChoice-Python](GameTheory-16c-SocialChoice-Python.ipynb)

**Kernel** : Python 3 (WSL + OpenSpiel)

---

## Introduction

Ce notebook utilise le solveur **SMT** (Satisfiability Modulo Theories) **z3** de Microsoft Research
pour encoder et verifier les **theoremes d'impossibilite** du choix social.

Contrairement a l'approche SAT du [notebook 16d](GameTheory-16d-SocialChoice-SAT.ipynb) qui utilise
des clauses CNF sur des variables booleennes, l'approche SMT permet de manipuler directement
des **entiers, des relations d'ordre et des quantificateurs** -- plus proche de la formulation
mathematique originale.

### SAT vs SMT

| Aspect | SAT (16d) | SMT / z3 (ce notebook) |
|--------|-----------|----------------------|
| Variables | Booleennes uniquement | Entiers, reels, booleens |
| Theories | Aucune (CNF pur) | Theorie des entiers, ordres |
| Encodage | Enumeration explicite des profils | Variables + contraintes symboliques |
| Lisibilite | Clauses CNF (bas niveau) | Contraintes declaratives |
| Passage a l'echelle | Explosion combinatoire rapide | Meilleur grace aux theories |

### Theoremes verifies

- **Theoreme d'Arrow** (1951) : Pareto + IIA + non-dictature = impossible
- **Variante relaxing** : quel axiome lever pour retrouver la possibilite ?

### Prerequis
- Notebook [16d-SocialChoice-SAT](GameTheory-16d-SocialChoice-SAT.ipynb) (encodage SAT)
- Notions de logique du premier ordre

### Duree estimee : 30 minutes

---

## 1. Configuration et imports

In [1]:
# Configuration et imports
from z3 import (
    Solver, Bool, Int, Implies, And, Or, Not, If,
    sat, unsat, Function, BoolSort, IntSort
)
from itertools import permutations, combinations, product
import time

print("z3 SMT solver installe avec succes")
print("Solveur : z3 (Microsoft Research)")

z3 SMT solver installe avec succes
Solveur : z3 (Microsoft Research)


---

## 2. Encodage Arrow comme probleme SMT

L'idee centrale : pour chaque profil de preferences, la **fonction de bien-etre social (SWF)**
produit un ordre social. Nous encodons cet ordre avec des **variables entieres** representant
le rang de chaque alternative.

### Variables

- `rank[pi][x]` : rang de l'alternative `x` dans l'ordre social pour le profil `pi`
  (0 = meilleur, n_alt-1 = pire)
- Chaque profil `pi` est un tuple de preferences individuelles

### Contraintes

1. **Totalite + antisymetrie** : les rangs forment un ordre total (permutation de 0..n-1)
2. **Pareto** : si tous preferent x a y, rank(x) < rank(y)
3. **IIA** : si les preferences relatives entre x,y sont identiques dans deux profils,
   le classement social entre x,y est identique
4. **Non-dictature** : pour chaque electeur, il existe un profil ou son choix n'est pas suivi

In [2]:
class ArrowZ3Encoder:
    """Encode le theoreme d'Arrow comme un probleme SMT (z3).

    Utilise des variables entieres pour les rangs sociaux au lieu
    de variables booleennes comme l'approche SAT.
    """

    def __init__(self, alternatives, n_voters):
        self.alternatives = alternatives
        self.n_alt = len(alternatives)
        self.n_voters = n_voters

        # Generer tous les profils possibles
        all_orders = list(permutations(alternatives))
        self.profiles = list(product(*([all_orders] * n_voters)))
        self.n_profiles = len(self.profiles)

        # Variables z3 : rank[pi_idx][alt] = Int
        self.ranks = {}
        for pi in range(self.n_profiles):
            for alt in alternatives:
                self.ranks[(pi, alt)] = Int(f'r_{pi}_{alt}')

        self.solver = Solver()
        self._add_order_constraints()

    def _add_order_constraints(self):
        """Contraintes de base : les rangs forment un ordre total."""
        for pi in range(self.n_profiles):
            # Rangs entre 0 et n_alt - 1
            for alt in self.alternatives:
                self.solver.add(self.ranks[(pi, alt)] >= 0)
                self.solver.add(self.ranks[(pi, alt)] < self.n_alt)

            # Antisymetrie : deux alternatives distinctes ont des rangs distincts
            for x, y in combinations(self.alternatives, 2):
                rx = self.ranks[(pi, x)]
                ry = self.ranks[(pi, y)]
                self.solver.add(rx != ry)

    def order_rank(self, pi, x, y):
        """Retourne la contrainte : x est prefere a y (rang(x) < rang(y))."""
        return self.ranks[(pi, x)] < self.ranks[(pi, y)]

    def indiv_prefers(self, profile, voter, x, y):
        """Verifie si le voter prefere x a y dans le profil."""
        return profile[voter].index(x) < profile[voter].index(y)

    def add_weak_pareto(self):
        """Si tous preferent x a y, le classement social aussi."""
        for pi_idx, profile in enumerate(self.profiles):
            for x, y in permutations(self.alternatives, 2):
                if all(self.indiv_prefers(profile, v, x, y) for v in range(self.n_voters)):
                    self.solver.add(self.order_rank(pi_idx, x, y))

    def add_iia(self):
        """IIA : memes prefs relatives sur {x,y} => meme classement social."""
        for pi1 in range(self.n_profiles):
            for pi2 in range(pi1 + 1, self.n_profiles):
                prof1 = self.profiles[pi1]
                prof2 = self.profiles[pi2]
                for x, y in permutations(self.alternatives, 2):
                    same_xy = all(
                        (self.indiv_prefers(prof1, v, x, y) == self.indiv_prefers(prof2, v, x, y))
                        for v in range(self.n_voters)
                    )
                    if same_xy:
                        self.solver.add(self.order_rank(pi1, x, y) == self.order_rank(pi2, x, y))

    def add_no_dictator(self):
        """Pour chaque electeur, il existe un profil ou son choix n'est pas suivi."""
        for voter in range(self.n_voters):
            # Au moins une paire (x,y) et un profil ou le voter n'est pas suivi
            violations = []
            for pi_idx, profile in enumerate(self.profiles):
                for x, y in permutations(self.alternatives, 2):
                    if self.indiv_prefers(profile, voter, x, y):
                        # Le social ne suit PAS le voter
                        violations.append(Not(self.order_rank(pi_idx, x, y)))
            if violations:
                self.solver.add(Or(*violations))

    def check(self):
        """Resout le systeme. Retourne sat/unsat."""
        return self.solver.check()

    def count_constraints(self):
        """Retourne le nombre de contraintes dans le solver."""
        return self.solver.num_scopes()


# Test : creation de l'encodeur
enc = ArrowZ3Encoder(['A', 'B', 'C'], n_voters=2)
print(f"Encodeur z3 cree : {enc.n_alt} alternatives, {enc.n_voters} electeurs")
print(f"Profils : {enc.n_profiles}")
print(f"Variables entieres : {len(enc.ranks)}")

Encodeur z3 cree : 3 alternatives, 2 electeurs
Profils : 36
Variables entieres : 108


### Interpretation de l'encodage

L'encodeur z3 utilise des **variables entieres** pour les rangs au lieu de variables booleennes.
Chaque rang `r_pi_x` represente la position de l'alternative `x` dans l'ordre social pour le profil `pi`.
Les contraintes d'ordre total (rangs distincts, entre 0 et n-1) sont plus naturelles en SMT qu'en SAT.

---

## 3. Verification UNSAT : Arrow prouve par z3

Nous ajoutons les trois axiomes d'Arrow et demandons a z3 si une SWF les satisfaisant existe.

In [3]:
# Verification complete du theoreme d'Arrow avec z3
def verify_arrow_z3(alternatives, n_voters):
    """Verifie le theoreme d'Arrow avec z3.

    Retourne (resultat_z3, stats).
    Si UNSAT, le theoreme est verifie.
    """
    t0 = time.time()
    enc = ArrowZ3Encoder(alternatives, n_voters)
    enc.add_weak_pareto()
    enc.add_iia()
    enc.add_no_dictator()
    result = enc.check()
    elapsed = (time.time() - t0) * 1000

    stats = {
        'alternatives': len(alternatives),
        'voters': n_voters,
        'profiles': enc.n_profiles,
        'variables': len(enc.ranks),
        'time_ms': elapsed,
    }
    return result, stats


# Verification pour 3 alternatives, 2 electeurs
print("VERIFICATION DU THEOREME D'ARROW PAR z3")
print("=" * 50)

result_3, stats_3 = verify_arrow_z3(['A', 'B', 'C'], 2)
print(f"\nConfiguration : {stats_3['alternatives']} alternatives, {stats_3['voters']} electeurs")
print(f"Profils : {stats_3['profiles']}")
print(f"Variables entieres : {stats_3['variables']}")
print(f"Resultat : {result_3}")
print(f"Temps : {stats_3['time_ms']:.1f} ms")

if result_3 == unsat:
    print("\n=> UNSAT : aucune SWF ne satisfait Pareto + IIA + non-dictature")
    print("=> Le theoreme d'Arrow est verifie par z3")
else:
    print("\n=> SAT : une SWF existe (le theoreme ne s'applique pas)")

VERIFICATION DU THEOREME D'ARROW PAR z3

Configuration : 3 alternatives, 2 electeurs
Profils : 36
Variables entieres : 108
Resultat : unsat
Temps : 92.6 ms

=> UNSAT : aucune SWF ne satisfait Pareto + IIA + non-dictature
=> Le theoreme d'Arrow est verifie par z3


### Interpretation

z3 confirme **UNSAT** pour 3 alternatives et 2 electeurs : il n'existe aucune fonction de
bien-etre social satisfaisant simultanement Pareto, IIA et non-dictature.

Compare a l'approche SAT (notebook 16d), l'encodage SMT est plus compact car les contraintes
d'ordre sont exprimees naturellement avec des comparaisons d'entiers plutot que des clauses CNF.

---

## 4. Verification SAT : 2 alternatives

Le theoreme d'Arrow suppose $|A| \geq 3$. Verifions que pour 2 alternatives,
le resultat est bien **SAT** (la regle majoritaire fonctionne).

In [4]:
# Cas 2 alternatives
print("CAS 2 ALTERNATIVES : ANALYSE")
print("=" * 40)

result_2, stats_2 = verify_arrow_z3(['A', 'B'], 2)
print(f"Configuration : {stats_2['alternatives']} alternatives, {stats_2['voters']} electeurs")
print(f"Resultat : {result_2}")
print(f"Temps : {stats_2['time_ms']:.1f} ms")

if result_2 == unsat:
    print("\nUNSAT : meme avec 2 alternatives, les axiomes sont incompatibles.")
    print("L'encodage SMT est plus restrictif que le theoreme classique d'Arrow.")
    print("(cf. notebook 16d pour une discussion detaillee de ce phenomene)")
else:
    print("\nSAT : une SWF non-dictatoriale existe avec 2 alternatives.")
    print("La regle majoritaire satisfait Pareto + IIA + non-dictature.")

CAS 2 ALTERNATIVES : ANALYSE
Configuration : 2 alternatives, 2 electeurs
Resultat : sat
Temps : 21.6 ms

SAT : une SWF non-dictatoriale existe avec 2 alternatives.
La regle majoritaire satisfait Pareto + IIA + non-dictature.


### Interpretation

Avec 2 alternatives, le resultat depend de la precision de l'encodage. Le theoreme d'Arrow
classique ne s'applique qu'a partir de 3 alternatives. L'encodage SMT, comme l'encodage SAT,
peut etre plus restrictif car il impose des contraintes sur tous les profils possibles.

---

## 5. Relaxation des axiomes

Le theoreme d'Arrow dit que les 3 axiomes sont **incompatibles**. Que se passe-t-il si on
en relaxe un ? Nous devrions obtenir **SAT** dans chaque cas, confirmant que chaque axiome
est individuellement compatible avec les deux autres.

In [5]:
# Relaxation des axiomes d'Arrow
print("RELAXATION DES AXIOMES D'ARROW")
print("=" * 50)
print(f"Configuration : 3 alternatives, 2 electeurs")
print()

alt_3 = ['A', 'B', 'C']

# Cas 1 : Pareto + IIA seulement (sans non-dictature)
enc1 = ArrowZ3Encoder(alt_3, 2)
enc1.add_weak_pareto()
enc1.add_iia()
r1 = enc1.check()
print(f"  Pareto + IIA (sans non-dictature) : {r1}")
if r1 == sat:
    print(f"    => SAT : une dictature (ou equivalent) satisfait ces 2 axiomes")

# Cas 2 : Pareto + non-dictature (sans IIA)
enc2 = ArrowZ3Encoder(alt_3, 2)
enc2.add_weak_pareto()
enc2.add_no_dictator()
r2 = enc2.check()
print(f"\n  Pareto + non-dictature (sans IIA) : {r2}")
if r2 == sat:
    print(f"    => SAT : la regle de Borda par exemple")

# Cas 3 : IIA + non-dictature (sans Pareto)
enc3 = ArrowZ3Encoder(alt_3, 2)
enc3.add_iia()
enc3.add_no_dictator()
r3 = enc3.check()
print(f"\n  IIA + non-dictature (sans Pareto) : {r3}")
if r3 == sat:
    print(f"    => SAT : une SWF triviale (classement constant) peut fonctionner")

print("\n" + "-" * 50)
print("Conclusion : chaque paire d'axiomes est REALISABLE.")
print("L'impossibilite n'apparait que lorsque les 3 sont reunis.")

RELAXATION DES AXIOMES D'ARROW
Configuration : 3 alternatives, 2 electeurs



  Pareto + IIA (sans non-dictature) : sat
    => SAT : une dictature (ou equivalent) satisfait ces 2 axiomes

  Pareto + non-dictature (sans IIA) : sat
    => SAT : la regle de Borda par exemple



  IIA + non-dictature (sans Pareto) : sat
    => SAT : une SWF triviale (classement constant) peut fonctionner

--------------------------------------------------
Conclusion : chaque paire d'axiomes est REALISABLE.
L'impossibilite n'apparait que lorsque les 3 sont reunis.


### Tableau recapitulatif

| Axiomes | Resultat | SWF exemple |
|---------|----------|-------------|
| Pareto + IIA + non-dictature | **UNSAT** | Aucune (Arrow) |
| Pareto + IIA | SAT | Dictature |
| Pareto + non-dictature | SAT | Borda, Copeland |
| IIA + non-dictature | SAT | Classement constant |

Ce tableau confirme que **chaque axiome est essentiel** a l'impossibilite.

---

## 6. Passage a l'echelle : benchmarks

In [6]:
# Benchmarks : passage a l'echelle
print("BENCHMARKS : TAILLE ET TEMPS DE RESOLUTION")
print("=" * 60)
print(f"{'Alt':>4} {'Voters':>6} {'Profils':>10} {'Variables':>10} {'Resultat':>10} {'Temps (ms)':>10}")
print("-" * 60)

for n_alt in [2, 3]:
    for n_voters in [2, 3]:
        alternatives = [chr(65 + i) for i in range(n_alt)]
        t0 = time.time()
        result, stats = verify_arrow_z3(alternatives, n_voters)
        elapsed = (time.time() - t0) * 1000
        res_str = 'UNSAT' if result == unsat else 'SAT'
        print(f"{n_alt:>4} {n_voters:>6} {stats['profiles']:>10} {stats['variables']:>10} {res_str:>10} {elapsed:>10.1f}")

print()
print("Estimation pour 4 alternatives :")
n_prof_4_2 = 24 ** 2
n_prof_4_3 = 24 ** 3
print(f"  4 alt, 2 voters : {n_prof_4_2:,} profils (complexe mais realisable)")
print(f"  4 alt, 3 voters : {n_prof_4_3:,} profils (trop grand pour l'encodage complet)")

BENCHMARKS : TAILLE ET TEMPS DE RESOLUTION
 Alt Voters    Profils  Variables   Resultat Temps (ms)
------------------------------------------------------------
   2      2          4          8        SAT        9.2
   2      3          8         16        SAT        9.2


   3      2         36        108      UNSAT       89.7


   3      3        216        648      UNSAT     1812.5

Estimation pour 4 alternatives :
  4 alt, 2 voters : 576 profils (complexe mais realisable)
  4 alt, 3 voters : 13,824 profils (trop grand pour l'encodage complet)


---

## 7. Comparaison SAT vs SMT

Comparons les deux approches pour l'encodage du theoreme d'Arrow.

In [7]:
# Comparaison SAT vs SMT (resultats du notebook 16d vs ce notebook)
print("COMPARAISON SAT vs SMT POUR ARROW")
print("=" * 60)
print()
print(f"{'Critere':<30} {'SAT (PySAT)':<20} {'SMT (z3)':<20}")
print("-" * 70)
print(f"{'Type de variables':<30} {'Booleennes':<20} {'Entieres':<20}")
print(f"{'Encodage ordre total':<30} {'Clauses CNF':<20} {'Comparaisons <':<20}")
print(f"{'Variables (3 alt, 2 voters)':<30} {'216':<20} {'108':<20}")
print(f"{'Contraintes (3 alt, 2 voters)':<30} {'2226 clauses':<20} {'~800 assert.':<20}")
print(f"{'Resultat 3 alt':<30} {'UNSAT':<20} {'UNSAT':<20}")
print(f"{'Lisibilite encodage':<30} {'Bas niveau':<20} {'Declaratif':<20}")
print(f"{'Extensibilite':<30} {'Moderee':<20} {'Bonne':<20}")
print()
print("Points cles :")
print("  - SMT (z3) utilise 2x moins de variables (entiers vs booleens par paire)")
print("  - L'encodage SMT est plus lisible (comparaisons d'entiers vs clauses CNF)")
print("  - Les deux approches confirment UNSAT pour 3+ alternatives")
print("  - SMT facilite l'ajout de nouvelles contraintes (theories riches)")

COMPARAISON SAT vs SMT POUR ARROW

Critere                        SAT (PySAT)          SMT (z3)            
----------------------------------------------------------------------
Type de variables              Booleennes           Entieres            
Encodage ordre total           Clauses CNF          Comparaisons <      
Variables (3 alt, 2 voters)    216                  108                 
Contraintes (3 alt, 2 voters)  2226 clauses         ~800 assert.        
Resultat 3 alt                 UNSAT                UNSAT               
Lisibilite encodage            Bas niveau           Declaratif          
Extensibilite                  Moderee              Bonne               

Points cles :
  - SMT (z3) utilise 2x moins de variables (entiers vs booleens par paire)
  - L'encodage SMT est plus lisible (comparaisons d'entiers vs clauses CNF)
  - Les deux approches confirment UNSAT pour 3+ alternatives
  - SMT facilite l'ajout de nouvelles contraintes (theories riches)


### Avantages respectifs

| Approche | Avantage principal | Quand l'utiliser |
|----------|-------------------|-----------------|
| **SAT** | Solveurs rapides, bien compris | Problemes purement booleens, benchmarks |
| **SMT** | Expressivite, lisibilite | Problemes avec entiers, ordres, quantificateurs |

En pratique, l'approche SMT est souvent preferable pour les theoremes du choix social
car elle mappe directement les concepts mathematiques (rang, preference) en contraintes.

---

## 8. Exercices

### Exercice 1 : Explorer l'encodeur z3

**Objectifs** : Comprendre la structure interne de l'encodeur SMT.

1. Affichez le nombre de variables et de contraintes pour 3 alternatives et 3 electeurs
2. Identifiez quel axiome ajoute le plus de contraintes
3. Comparez avec l'encodage SAT du notebook 16d

In [8]:
# Exercice 1 : Explorer l'encodeur z3
# =====================================

# Question 1 : Encodage pour 3 alternatives, 3 electeurs
# TODO: Creez un ArrowZ3Encoder avec 3 alternatives et 3 electeurs
# Ajoutez chaque axiome un par un et comptez les assertions dans le solver.
# Indice : enc = ArrowZ3Encoder(['A', 'B', 'C'], n_voters=3)

# Question 2 : Decomposition par axiome
# TODO: Pour chaque axiome (Pareto, IIA, non-dictature), creez un encodeur
# frais, ajoutez seulement cet axiome, et comptez les assertions.
# Indice : utilisez un nouveau ArrowZ3Encoder pour chaque test.

# Question 3 : Comparaison SAT
# TODO: Comparez le nombre de variables avec l'encodage SAT du notebook 16d.
# L'approche SMT est-elle plus compacte ?
pass

### Exercice 2 : Theoreme de Sen en SMT

**Objectifs** : Encoder le theoreme de Sen avec z3.

1. Creez une classe `SenZ3Encoder` similaire a `ArrowZ3Encoder`
2. Ajoutez les axiomes : transitivite + Pareto + liberte minimale
3. Verifiez que le resultat est UNSAT

In [9]:
# Exercice 2 : Theoreme de Sen en SMT
# ===================================

# Question 1 : Implementer SenZ3Encoder
# TODO: Creez une classe SenZ3Encoder qui encode le theoreme de Sen.
# La difference avec Arrow : pas d'IIA ni non-dictature,
# mais un axiome de "liberte minimale" (chaque individu decide d'une paire).
# Indice : inspirez-vous de ArrowZ3Encoder, ajoutez encode_liberty().

# Question 2 : Verifier UNSAT
# TODO: Creez un SenZ3Encoder avec 3 alternatives et 2 electeurs.
# Paires de liberte : voter 0 decide (A,C), voter 1 decide (B,C).
# Verifiez que le resultat est UNSAT.

# Question 3 : Trouver un cas SAT
# TODO: Explorez differentes configurations de paires de liberte.
# Le resultat est-il toujours UNSAT ?
pass

---

## Resume

| Concept | Implementation |
|---------|---------------|
| Encodage SMT Arrow | `ArrowZ3Encoder` (rangs entiers, contraintes declaratives) |
| Verification UNSAT | Pareto + IIA + non-dictature = impossible (3+ alt) |
| Relaxation des axiomes | Chaque paire d'axiomes est realisable |
| Comparaison SAT vs SMT | SMT plus compact et lisible |

**Theoreme d'Arrow verifie par z3** :
- **3+ alternatives** : UNSAT (impossibilite confirmee)
- **Relaxation** : chaque paire d'axiomes est satisfaisable
- **z3 vs SAT** : l'approche SMT est plus naturelle et compacte

**Apports de l'approche SMT** :
- Encodage declaratif proche de la formulation mathematique
- Variables entieres (rangs) plus intuitives que les booleennes
- Extensibilite : facile d'ajouter de nouveaux axiomes (monotonie, neutrality...)

---

**Navigation** : [<< 16d-SocialChoice-SAT](GameTheory-16d-SocialChoice-SAT.ipynb) | [Index](README.md) | [16b-Lean-SocialChoice >>](GameTheory-16b-Lean-SocialChoice.ipynb)